In [1]:
# =============================================
# Import packages and libraries
# =============================================
import numpy as np
import pandas as pd
import pyodbc
from sklearn.preprocessing import StandardScaler
import joblib


In [2]:
# =============================================
# Read data from SQL Server Database
# =============================================
conn = pyodbc.connect(
        "Driver={SQL Server};"
        "Server=LocalHost\\SQLEXPRESS;"
        "Database=BankChurn;"
        "Trusted_Connection=yes;"
    )
cursor = conn.cursor()



query = """
SELECT
    d.Gender, d.Age, d.Salary, l.Geography,
    a.Tenure, a.Balance, a.NumProducts, a.HasCreditCard, a.IsActive,
    d.Churned
FROM demographic d
JOIN account a ON a.CustomerId = d.CustomerId
JOIN [location] l ON l.LocationId = d.LocationId
"""

df = pd.read_sql(query,conn)

C:\Users\mayur\AppData\Local\Temp\ipykernel_17992\3768930031.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query,conn)


In [3]:
df

,Gender,Age,Salary,Geography,Tenure,Balance,NumProducts,HasCreditCard,IsActive,Churned
0,Female,39,90212.38,Spain,9,161993.89,1,False,False,False
1,Male,35,83256.26,Germany,1,119827.49,1,True,True,True
2,Male,47,96517.97,USA,1,64430.06,2,False,True,False
3,Female,30,188258.49,Spain,6,57809.32,1,True,False,False
4,Male,48,74510.65,Spain,10,96048.55,1,True,False,False
...,...,...,...,...,...,...,...,...,...,...
9995,Female,37,80215.20,UK,8,113899.92,1,False,False,False
9996,Male,37,175082.20,France,8,152303.66,2,True,True,False
9997,Female,39,199808.10,France,9,100116.67,1,True,True,True
9998,Female,34,60728.89,Spain,1,154139.45,1,True,True,False


In [4]:
# =============================================
# Split Train and Test Dataset
# =============================================
FRAC = 0.8
SEED = 200

df_train = df.sample(frac = FRAC, random_state = SEED)

df_test = df.drop(df_train.index)
print(len(df_train))
print(len(df_test))



8000
2000


In [5]:
# =============================================
# Feature Engineering in Training
# =============================================
df_train['BalanceSalaryRatio'] = df_train.Balance / df_train.Salary
df_train['TenureByAge'] = df_train.Tenure / df_train.Age

In [6]:
df_train

,Gender,Age,Salary,Geography,Tenure,Balance,NumProducts,HasCreditCard,IsActive,Churned,BalanceSalaryRatio,TenureByAge
8159,Male,35,64050.19,USA,0,133377.80,1,False,True,False,2.082395,0.000000
6332,Male,20,150135.38,UK,3,119714.25,2,False,True,False,0.797375,0.150000
8895,Male,32,178331.36,Spain,1,132187.73,2,True,True,False,0.741248,0.031250
5351,Female,25,82393.08,UK,7,119827.49,2,False,True,False,1.454339,0.280000
4314,Male,37,77108.66,Germany,9,119827.49,2,False,True,False,1.554008,0.243243
...,...,...,...,...,...,...,...,...,...,...,...,...
7554,Female,32,126517.46,France,4,109421.13,2,True,True,False,0.864870,0.125000
3077,Female,29,23237.25,UK,7,119827.49,2,True,False,False,5.156698,0.241379
1898,Male,34,102299.81,Spain,0,151839.26,1,True,False,True,1.484257,0.000000
5046,Male,37,166678.28,Germany,10,106242.67,1,True,True,False,0.637412,0.270270


In [7]:
# =============================================
# Scaling Numerical Features in Training
# =============================================
num_cols = df_train.select_dtypes(include=['int64','float64']).columns
scaler = StandardScaler()
df_train[num_cols] = scaler.fit_transform(df_train[num_cols])


In [20]:
df_train.select_dtypes(include=["int64" , "float64"]).describe()

,Age,Salary,Tenure,Balance,NumProducts,BalanceSalaryRatio,TenureByAge,Gender_Female,Gender_Male,Geography_Canada,Geography_France,Geography_Germany,Geography_Spain,Geography_UK,Geography_USA,HasCreditCard_False,HasCreditCard_True,IsActive_False,IsActive_True
count,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8.000000e+03,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,2.078338e-16,4.973799e-17,-1.341149e-16,4.884981e-18,1.731948e-16,7.105427e-18,-3.907985e-17,0.457375,0.542625,0.165750,0.172000,0.165375,0.162250,0.171000,0.163625,0.295500,0.704500,0.484625,0.515375
std,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,1.000063e+00,0.498211,0.498211,0.371879,0.377404,0.371542,0.368703,0.376532,0.369958,0.456296,0.456296,0.499795,0.499795
min,-1.985493e+00,-1.745613e+00,-1.730806e+00,-4.406419e+00,-9.072672e-01,-5.032722e-02,-1.534952e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-6.571423e-01,-8.456748e-01,-1.040932e+00,-3.459694e-01,-9.072672e-01,-4.503575e-02,-8.185057e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-1.827313e-01,-6.253918e-04,-6.122625e-03,7.758383e-03,-9.072672e-01,-4.173633e-02,-1.020592e-01,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000
75%,4.814441e-01,8.576727e-01,1.028687e+00,3.230496e-01,8.025414e-01,-3.259185e-02,6.860319e-01,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
max,5.035790e+00,1.736142e+00,1.718560e+00,5.488462e+00,4.222159e+00,8.589120e+01,4.634448e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [8]:
# =============================================
# Encoding Categorical Features in Training
# =============================================
cat_cols = df_train.select_dtypes(
    include=['object', 'string', 'bool']
).columns.drop(['Churned'])

df_train = pd.get_dummies(
    df_train,
    columns=cat_cols,
    dtype=int
)

In [23]:
df_train

,Age,Salary,Tenure,Balance,NumProducts,Churned,BalanceSalaryRatio,TenureByAge,Gender_Female,Gender_Male,Geography_Canada,Geography_France,Geography_Germany,Geography_Spain,Geography_UK,Geography_USA,HasCreditCard_False,HasCreditCard_True,IsActive_False,IsActive_True
8159,-0.372496,-0.630673,-1.730806,0.574363,-0.907267,False,-0.034587,-1.534952,0,1,0,0,0,0,0,1,1,0,0,1
6332,-1.795729,0.868108,-0.695996,0.003023,0.802541,False,-0.044991,0.130786,0,1,0,0,0,0,1,0,1,0,0,1
8895,-0.657142,1.359012,-1.385869,0.524601,0.802541,False,-0.045446,-1.187923,0,1,0,0,0,1,0,0,0,1,0,1
5351,-1.321318,-0.311315,0.683751,0.007758,0.802541,False,-0.039672,1.574426,1,0,0,0,0,0,1,0,1,0,0,1
4314,-0.182731,-0.403319,1.373624,0.007758,0.802541,False,-0.038865,1.166245,0,1,0,0,1,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7554,-0.657142,0.456910,-0.351059,-0.427383,0.802541,False,-0.044445,-0.146837,1,0,0,1,0,0,0,0,0,1,0,1
3077,-0.941789,-1.341244,0.683751,0.007758,0.802541,False,-0.009696,1.145546,1,0,0,0,0,0,1,0,0,1,1,0
1898,-0.467378,0.035270,-1.730806,1.346327,-0.907267,True,-0.039430,-1.534952,0,1,0,0,0,1,0,0,0,1,1,0
5046,-0.182731,1.156127,1.718560,-0.560289,-0.907267,False,-0.046287,1.466378,0,1,0,0,1,0,0,0,0,1,0,1


In [ ]:
# =============================================
# Preprocessing Pipeline in Testing
# =============================================
def DfTestPipeline(df_test):
    #feature engg
    df_test['BalanceSalaryRatio'] = df_test.Balance / df_test.Salary
    df_test['TenureByAge'] = df_test.Tenure / df_test.Age
    
    #scaling
    df_test[num_cols] = scaler.transform(df_test[num_cols])

    #encoding
    df_test = pd.get_dummies(
    df_test,
    columns=cat_cols,
    dtype=int)

   

    return df_test

df_test = DfTestPipeline(df_test)

In [12]:
# =============================================
# Re-order columns in Testing and Training
# =============================================
df_train.columns.equals(df_test.columns)
df_test = df_test[df_train.columns]


In [13]:
df_test

,Age,Salary,Tenure,Balance,NumProducts,Churned,BalanceSalaryRatio,TenureByAge,Gender_Female,Gender_Male,Geography_Canada,Geography_France,Geography_Germany,Geography_Spain,Geography_UK,Geography_USA,HasCreditCard_False,HasCreditCard_True,IsActive_False,IsActive_True
2,0.766091,-0.065395,-1.385869,-2.308680,0.802541,False,-0.046043,-1.298677,0,1,0,0,0,0,0,1,1,0,0,1
3,-0.846907,1.531848,0.338814,-2.585525,-0.907267,False,-0.048961,0.686032,1,0,0,0,0,1,0,0,0,1,1,0
5,-0.087849,-1.213335,1.373624,0.007758,0.802541,False,-0.019725,1.095161,0,1,0,0,0,0,0,1,1,0,1,0
17,0.481444,0.811623,1.373624,0.993067,-0.907267,True,-0.043544,0.736509,0,1,0,0,0,0,1,0,1,0,1,0
18,-1.321318,-0.922164,-0.006123,0.007758,0.802541,False,-0.030939,0.686032,0,1,0,0,0,0,0,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9951,-0.562260,-0.980930,0.338814,2.311142,-0.907267,False,-0.019212,0.484124,0,1,0,0,0,1,0,0,0,1,0,1
9967,1.240502,1.315701,-0.695996,0.007758,-0.907267,True,-0.045930,-0.894284,0,1,0,0,0,0,1,0,0,1,1,0
9969,0.101915,0.369626,1.718560,0.246397,-0.907267,True,-0.043082,1.241278,1,0,0,0,0,0,0,1,0,1,1,0
9973,0.955855,1.074896,1.028687,0.007758,0.802541,False,-0.045459,0.278096,0,1,0,0,0,1,0,0,0,1,1,0


In [14]:
# =============================================
# Declare Features and Targets
# =============================================
df_train_x = df_train.drop('Churned', axis=1)
df_train_y = df_train['Churned']

df_test_x = df_test.drop('Churned', axis=1)
df_test_y = df_test['Churned']


In [15]:
df_test_x

,Age,Salary,Tenure,Balance,NumProducts,BalanceSalaryRatio,TenureByAge,Gender_Female,Gender_Male,Geography_Canada,Geography_France,Geography_Germany,Geography_Spain,Geography_UK,Geography_USA,HasCreditCard_False,HasCreditCard_True,IsActive_False,IsActive_True
2,0.766091,-0.065395,-1.385869,-2.308680,0.802541,-0.046043,-1.298677,0,1,0,0,0,0,0,1,1,0,0,1
3,-0.846907,1.531848,0.338814,-2.585525,-0.907267,-0.048961,0.686032,1,0,0,0,0,1,0,0,0,1,1,0
5,-0.087849,-1.213335,1.373624,0.007758,0.802541,-0.019725,1.095161,0,1,0,0,0,0,0,1,1,0,1,0
17,0.481444,0.811623,1.373624,0.993067,-0.907267,-0.043544,0.736509,0,1,0,0,0,0,1,0,1,0,1,0
18,-1.321318,-0.922164,-0.006123,0.007758,0.802541,-0.030939,0.686032,0,1,0,0,0,0,0,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9951,-0.562260,-0.980930,0.338814,2.311142,-0.907267,-0.019212,0.484124,0,1,0,0,0,1,0,0,0,1,0,1
9967,1.240502,1.315701,-0.695996,0.007758,-0.907267,-0.045930,-0.894284,0,1,0,0,0,0,1,0,0,1,1,0
9969,0.101915,0.369626,1.718560,0.246397,-0.907267,-0.043082,1.241278,1,0,0,0,0,0,0,1,0,1,1,0
9973,0.955855,1.074896,1.028687,0.007758,0.802541,-0.045459,0.278096,0,1,0,0,0,1,0,0,0,1,1,0


In [16]:
df_test_y

2       False
3       False
5       False
17       True
18      False
        ...  
9951    False
9967     True
9969     True
9973    False
9988     True
Name: Churned, Length: 2000, dtype: bool

In [17]:
# =============================================
# Save Data
# =============================================
artifacts = {
    "X_train": df_train_x,
    "y_train": df_train_y,
    "X_test": df_test_x,
    "y_test": df_test_y
}

joblib.dump(artifacts, 'dataset_bundle.pkl')

['dataset_bundle.pkl']